# Rule Support & Confidence Counting — Benchmark

Compiles and benchmarks three implementations:
1. **Rewritten C++** — faithful rewrite of original Scala logic (baseline)
2. **Optimized CPU** — cache-friendly CSR layout, stack-local DFS
3. **GPU (CUDA)** — CSR + `__constant__` memory + bitmap body-size counting

Works on:
- **Google Colab** (with or without GPU runtime)
- **VSE JupyterHub** (CPU only — GPU cell is skipped automatically)

In [1]:
import os
os.chdir(os.path.expanduser("~"))
!ls -lah

total 60K
drwxr-x--- 12 jupyter-naav00@vse.cz jupyter-naav00@vse.cz 4.0K Apr 25 15:46 .
drwxr-xr-x 28 root                  root                  4.0K Apr 30 20:00 ..
-rw-r--r--  1 jupyter-naav00@vse.cz jupyter-naav00@vse.cz  220 Mar 31  2024 .bash_logout
-rw-r--r--  1 jupyter-naav00@vse.cz jupyter-naav00@vse.cz 3.7K Mar 31  2024 .bashrc
lrwxrwxrwx  1 jupyter-naav00@vse.cz jupyter-naav00@vse.cz   14 Feb  3 18:27 exchange -> /srv/exchange/
drwxr-xr-x  2 jupyter-naav00@vse.cz jupyter-naav00@vse.cz 4.0K Apr  9 21:25 .ipynb_checkpoints
drwxr-xr-x  3 jupyter-naav00@vse.cz jupyter-naav00@vse.cz 4.0K Apr  9 12:39 .ipython
drwxr-xr-x  4 jupyter-naav00@vse.cz jupyter-naav00@vse.cz 4.0K Apr 25 10:45 .jupyter
drwxr-xr-x  3 jupyter-naav00@vse.cz jupyter-naav00@vse.cz 4.0K Apr  9 12:39 .local
drwxr-xr-x 12 jupyter-naav00@vse.cz jupyter-naav00@vse.cz 4.0K Apr 14 08:18 MasterThesis
drwxr-xr-x 10 jupyter-naav00@vse.cz jupyter-naav00@vse.cz 4.0K Apr  9 21:28 MasterThesis_01_OLD
drwxr-xr-x  8 jupyter-na

In [2]:
import os, subprocess, sys, time, json, re
from pathlib import Path

# =============================================================================
# CHANGE THIS to wherever you uploaded / cloned the MasterThesisFinal folder.
#   Colab default : /content/drive/MyDrive/MasterThesisFinal
#   VSE Jupyter   : /home/jupyter-naav00@vse.cz/MasterThesisFinal
BASE_PATH = '/home/jupyter-naav00@vse.cz/MasterThesisFinal'

# =============================================================================

BASE = Path(BASE_PATH)
if not BASE.exists():
    raise FileNotFoundError(
        f'Folder not found: {BASE}\n'
        'Update BASE_PATH above to point at your MasterThesisFinal folder.'
    )
if not (BASE / 'GPU_03').exists():
    raise FileNotFoundError(
        f'Cannot find GPU_03 inside {BASE}\n'
        'Are you pointing at the right folder?'
    )

IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

def has_command(cmd):
    return subprocess.run(['which', cmd], capture_output=True).returncode == 0

HAS_NVCC = has_command('nvcc')
HAS_GPU  = False
if HAS_NVCC:
    r = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                       capture_output=True, text=True)
    HAS_GPU = r.returncode == 0 and r.stdout.strip() != ''

print(f'Environment : {"Google Colab" if IS_COLAB else "JupyterHub / local"}')
print(f'nvcc found  : {HAS_NVCC}')
print(f'GPU found   : {HAS_GPU}')
if HAS_GPU:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap',
                          '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())

DIR_REWRITTEN = BASE / 'Rewritten_CPP'
DIR_CPU       = BASE / 'CPU_01_nonparallel'
DIR_GPU       = BASE / 'GPU_03'
DATA_DIR      = BASE / 'body_test_data'

TTL_FILE  = DATA_DIR / 'gpu-test-100k.ttl'
RULE_FILE = DATA_DIR / 'rule.txt'

RULE_FILE_JSON = DATA_DIR / 'rule.json'
print(f'\nBase dir  : {BASE}')
print(f'TTL file  : {TTL_FILE}  (exists={TTL_FILE.exists()})')
print(f'Rules txt : {RULE_FILE} (exists={RULE_FILE.exists()})')
print(f'Rules json: {RULE_FILE_JSON} (exists={RULE_FILE_JSON.exists()})')

Environment : JupyterHub / local
nvcc found  : True
GPU found   : True
NVIDIA H200 NVL, 9.0
NVIDIA H200 NVL, 9.0
NVIDIA H200 NVL, 9.0
NVIDIA H200 NVL, 9.0

Base dir  : /home/jupyter-naav00@vse.cz/MasterThesisFinal
TTL file  : /home/jupyter-naav00@vse.cz/MasterThesisFinal/body_test_data/gpu-test-100k.ttl  (exists=True)
Rules txt : /home/jupyter-naav00@vse.cz/MasterThesisFinal/body_test_data/rule.txt (exists=True)
Rules json: /home/jupyter-naav00@vse.cz/MasterThesisFinal/body_test_data/rule.json (exists=True)


In [3]:
# -- Helper: run a shell command and print output -------------------------------
def run(cmd, cwd=None, check=True):
    print(f'$ {cmd}')
    result = subprocess.run(cmd, shell=True, cwd=cwd or BASE,
                            capture_output=True, text=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr, file=sys.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed (exit {result.returncode})')
    return result

# -- Helper: run a benchmark binary and return timing (ms) ---------------------
def benchmark(cmd, cwd=None, runs=3):
    """Run cmd `runs` times, return (min_ms, avg_ms, all stdout)."""
    times = []
    last_stdout = ''
    for i in range(runs):
        t0 = time.perf_counter()
        r  = subprocess.run(cmd, shell=True, cwd=cwd or BASE,
                            capture_output=True, text=True)
        t1 = time.perf_counter()
        if r.returncode != 0:
            raise RuntimeError(f'Benchmark failed: {r.stderr}')
        times.append((t1 - t0) * 1000)
        last_stdout = r.stdout
    return min(times), sum(times)/len(times), last_stdout

# -- Helper: extract total_ms from human-readable output -----------------------
def extract_total_ms(stdout):
    m = re.search(r'Total wall time:\s+([0-9.]+)\s*ms', stdout)
    if m: return float(m.group(1))
    m = re.search(r'Total time:\s+([0-9.]+)\s*ms', stdout)
    if m: return float(m.group(1))
    return None


import psutil, threading

def get_vram_mb():
    r = subprocess.run(
        ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
        capture_output=True, text=True)
    if r.returncode != 0: return 0
    return int(r.stdout.strip().split('\n')[0])

def measure_peak_memory(cmd, cwd=None, interval=0.5):
    baseline_ram  = psutil.virtual_memory().used / (1024*1024)
    baseline_vram = get_vram_mb()
    peak_ram  = [baseline_ram]
    peak_vram = [baseline_vram]
    stop_flag = [False]
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd or BASE,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    def sampler():
        while not stop_flag[0]:
            peak_ram[0]  = max(peak_ram[0],  psutil.virtual_memory().used / (1024*1024))
            peak_vram[0] = max(peak_vram[0], get_vram_mb())
            time.sleep(interval)
    t = threading.Thread(target=sampler, daemon=True)
    t0 = time.perf_counter()
    t.start()
    for line in proc.stdout:
        pass  # drain output silently
    proc.wait()
    stop_flag[0] = True
    t.join()
    wall_ms = (time.perf_counter() - t0) * 1000
    return peak_ram[0] - baseline_ram, peak_vram[0] - baseline_vram, wall_ms


# -- Helper: load body-atom counts from JSON rule file -------------------------
def load_rule_atom_counts(rule_json_path):
    with open(rule_json_path) as _f:
        _rules = json.load(_f)
    return {i + 1: len(r['body']) for i, r in enumerate(_rules)}

# -- Helper: parse per-rule timing from Rewritten/CPU stdout -------------------
def parse_cpu_output(stdout):
    records = []
    for line in stdout.splitlines():
        m = re.search(r'supportMs=([0-9.]+)\s+confidenceMs=([0-9.]+)', line)
        if m:
            records.append({'supportMs': float(m.group(1)), 'confidenceMs': float(m.group(2))})
    for i, r in enumerate(records):
        r['ruleId'] = i + 1
    return records

# -- Helper: parse per-rule timing from GPU stdout -----------------------------
def parse_gpu_output(stdout):
    records = []
    for line in stdout.splitlines():
        m = re.search(r'Rule\s+(\d+):.*SupportTime:\s*([0-9.]+)\s*ms.*ConfidenceTime:\s*([0-9.]+)\s*ms', line)
        if m:
            records.append({'ruleId': int(m.group(1)),
                            'supportMs': float(m.group(2)),
                            'confidenceMs': float(m.group(3))})
    return records

# -- Helper: print per-body-atom breakdown -------------------------------------
def print_body_stats(records, atom_counts, label):
    from collections import defaultdict
    g_sup  = defaultdict(list)
    g_conf = defaultdict(list)
    for r in records:
        n = atom_counts.get(r['ruleId'], '?')
        g_sup[n].append(r['supportMs'])
        g_conf[n].append(r['confidenceMs'])
    print(f'\n-- Per-body-atom breakdown: {label} --')
    print(f'  {"Atoms":<8} {"Rules":>6} {"Avg SupportMs":>15} {"Avg ConfMs":>12}')
    for n in sorted(k for k in g_sup if isinstance(k, int)):
        avg_s = sum(g_sup[n]) / len(g_sup[n])
        avg_c = sum(g_conf[n]) / len(g_conf[n])
        print(f'  {n:<8} {len(g_sup[n]):>6} {avg_s:>15.3f} {avg_c:>12.3f}')

print('Helpers ready.')


Helpers ready.


---
## Step 1 — Compile all implementations

In [4]:
# -- Compile: Rewritten C++ (baseline) ----------------------------------------
print('Compiling Rewritten_CPP ...')
run(
    'g++ -O3 -std=c++17 '
    'main.cpp FinalRule.cpp RdfIndexes.cpp RuleParser.cpp SupportCounting.cpp '
    '-o rdf_rules_rewritten',
    cwd=DIR_REWRITTEN
)
print('OK — binary: Rewritten_CPP/rdf_rules_rewritten')

Compiling Rewritten_CPP ...
$ g++ -O3 -std=c++17 main.cpp FinalRule.cpp RdfIndexes.cpp RuleParser.cpp SupportCounting.cpp -o rdf_rules_rewritten
OK — binary: Rewritten_CPP/rdf_rules_rewritten


In [5]:
# -- Compile: Optimized CPU ----------------------------------------------------
print('Compiling CPU_01_nonparallel ...')
run(
    'g++ -O3 -std=c++17 '
    'main.cpp FinalRule.cpp RdfIndexes.cpp RuleParser.cpp SupportCounting.cpp '
    '-o rdf_rules_test',
    cwd=DIR_CPU
)
print('OK — binary: CPU_01_nonparallel/rdf_rules_test')

Compiling CPU_01_nonparallel ...
$ g++ -O3 -std=c++17 main.cpp FinalRule.cpp RdfIndexes.cpp RuleParser.cpp SupportCounting.cpp -o rdf_rules_test
OK — binary: CPU_01_nonparallel/rdf_rules_test


In [6]:
# -- Compile: GPU (only if CUDA available) ------------------------------------
if not HAS_NVCC:
    print('nvcc not found — skipping GPU compilation.')
    print('This is expected on VSE JupyterHub and CPU-only Colab runtimes.')
else:
    print('Compiling GPU_03 benchmark binary ...')
    run(
        'nvcc -O3 -std=c++17 -Wno-deprecated-gpu-targets '
        'bench_gpu.cu FinalRule.cpp RdfIndexes.cpp RuleParser.cpp '
        '-o bench_gpu',
        cwd=DIR_GPU
    )
    print('OK — binary: GPU_03/bench_gpu')

    print('\nCompiling GPU_03 shared library (for Scala/JNA integration) ...')
    run(
        'nvcc -O3 -std=c++17 -Wno-deprecated-gpu-targets -shared -Xcompiler -fPIC '
        'gpu_rules_lib.cu FinalRule.cpp RdfIndexes.cpp RuleParser.cpp '
        '-o libgpu_rules.so',
        cwd=DIR_GPU
    )
    print('OK — library: GPU_03/libgpu_rules.so')

Compiling GPU_03 benchmark binary ...
$ nvcc -O3 -std=c++17 -Wno-deprecated-gpu-targets bench_gpu.cu FinalRule.cpp RdfIndexes.cpp RuleParser.cpp -o bench_gpu
OK — binary: GPU_03/bench_gpu

Compiling GPU_03 shared library (for Scala/JNA integration) ...
$ nvcc -O3 -std=c++17 -Wno-deprecated-gpu-targets -shared -Xcompiler -fPIC gpu_rules_lib.cu FinalRule.cpp RdfIndexes.cpp RuleParser.cpp -o libgpu_rules.so
OK — library: GPU_03/libgpu_rules.so


---
## Step 2 — Run benchmarks

Each implementation is run 3 times; we report the **fastest** run (avoids cold-start noise).
The test dataset is `body_test_data/gpu-test-100k.ttl` with 100 000 triples.

In [7]:
# -- Benchmark: Rewritten C++ --------------------------------------------------
print('Running Rewritten_CPP benchmark ...')
cmd_rewritten = f'{DIR_REWRITTEN}/rdf_rules_rewritten {TTL_FILE} {RULE_FILE_JSON}'
min_ms_r, avg_ms_r, out_r = benchmark(cmd_rewritten)
total_r = extract_total_ms(out_r)
records_r = parse_cpu_output(out_r)

print(out_r)
print(f'Wall time: min={min_ms_r:.0f} ms  avg={avg_ms_r:.0f} ms  (reported total={total_r} ms)')
atom_counts = load_rule_atom_counts(RULE_FILE_JSON)
print_body_stats(records_r, atom_counts, 'Rewritten C++')


Running Rewritten_CPP benchmark ...
=== Graph loaded ===
Triples: 100000
Unique RDF terms: 17527
Predicates in PSO: 1
Predicates in POS: 1

Rules loaded: 10

Rule 1: ( ?0 https://example.org/relation/interacts_with ?1 ) => ( ?1 https://example.org/relation/interacts_with ?0 )
  support=820  headCoverage=0.0082  bodySize=100000  confidence=0.0082  supportMs=12  confidenceMs=21
Rule 2: ( ?0 https://example.org/relation/interacts_with ?1 ) => ( ?0 https://example.org/relation/interacts_with ?1 )
  support=0  headCoverage=0  bodySize=0  confidence=0  supportMs=16  confidenceMs=0
Rule 3: ( ?0 https://example.org/relation/interacts_with ?1 ) ^ ( ?1 https://example.org/relation/interacts_with ?2 ) => ( ?0 https://example.org/relation/interacts_with ?2 )
  support=63336  headCoverage=0.63336  bodySize=584395  confidence=0.108379  supportMs=49  confidenceMs=128
Rule 4: ( ?0 https://example.org/relation/interacts_with ?1 ) ^ ( ?2 https://example.org/relation/interacts_with ?1 ) => ( ?0 https://e

In [8]:
# -- Benchmark: Optimized CPU --------------------------------------------------
print('Running CPU_01_nonparallel benchmark ...')
cmd_cpu = f'{DIR_CPU}/rdf_rules_test {TTL_FILE} {RULE_FILE_JSON}'
min_ms_c, avg_ms_c, out_c = benchmark(cmd_cpu)
total_c = extract_total_ms(out_c)
records_c = parse_cpu_output(out_c)

print(out_c)
print(f'Wall time: min={min_ms_c:.0f} ms  avg={avg_ms_c:.0f} ms  (reported total={total_c} ms)')
print_body_stats(records_c, atom_counts, 'Optimized CPU')


Running CPU_01_nonparallel benchmark ...
=== Graph loaded ===
Triples (deduped): 100000
Unique RDF terms: 17527
Predicates: 1

Rules loaded: 10

Rule 1: ( ?0 https://example.org/relation/interacts_with ?1 ) => ( ?1 https://example.org/relation/interacts_with ?0 )
  support=820  headCoverage=0.0082  bodySize=100000  confidence=0.0082  supportMs=4.96783  confidenceMs=5.26415
Rule 2: ( ?0 https://example.org/relation/interacts_with ?1 ) => ( ?0 https://example.org/relation/interacts_with ?1 )
  support=0  headCoverage=0  bodySize=100000  confidence=0  supportMs=10.3738  confidenceMs=6.1449
Rule 3: ( ?0 https://example.org/relation/interacts_with ?1 ) ^ ( ?1 https://example.org/relation/interacts_with ?2 ) => ( ?0 https://example.org/relation/interacts_with ?2 )
  support=63336  headCoverage=0.63336  bodySize=584395  confidence=0.108379  supportMs=46.6779  confidenceMs=52.1229
Rule 4: ( ?0 https://example.org/relation/interacts_with ?1 ) ^ ( ?2 https://example.org/relation/interacts_with ?

In [9]:
# -- Benchmark: GPU ------------------------------------------------------------
min_ms_g = avg_ms_g = total_g = None
records_g = []

if not HAS_GPU:
    print('No GPU available — skipping GPU benchmark.')
else:
    print('Running GPU_03 benchmark ...')
    cmd_gpu = f'{DIR_GPU}/bench_gpu {TTL_FILE} {RULE_FILE_JSON}'
    min_ms_g, avg_ms_g, out_g = benchmark(cmd_gpu)
    total_g = extract_total_ms(out_g)
    records_g = parse_gpu_output(out_g)
    print(out_g)
    print(f'Wall time: min={min_ms_g:.0f} ms  avg={avg_ms_g:.0f} ms  (reported total={total_g} ms)')
    print_body_stats(records_g, atom_counts, 'GPU CUDA')


Running GPU_03 benchmark ...
=== Index built ===
Triples (deduped): 100000
Unique RDF terms: 17527
Predicates: 1
Entity count: 17527
CSR index built: 1 predicates  (1673 ms)

=== Rules loaded ===
Rule count: 10

Body-size strategy: bitmap (36.6 MB)

Rule 1: ( ?0 https://example.org/relation/interacts_with ?1 ) => ( ?1 https://example.org/relation/interacts_with ?0 ) | Support: 820, HeadCoverage: 0.0082, HeadSupport: 100000, HeadSize: 100000, BodySize: 100000, Confidence: 0.0082, SupportTime: 0.04464 ms, ConfidenceTime: 0.064864 ms

Rule 2: ( ?0 https://example.org/relation/interacts_with ?1 ) => ( ?0 https://example.org/relation/interacts_with ?1 ) | Support: 100000, HeadCoverage: 1, HeadSupport: 100000, HeadSize: 100000, BodySize: 100000, Confidence: 1, SupportTime: 0.010848 ms, ConfidenceTime: 0.008512 ms

Rule 3: ( ?0 https://example.org/relation/interacts_with ?1 ) ^ ( ?1 https://example.org/relation/interacts_with ?2 ) => ( ?0 https://example.org/relation/interacts_with ?2 ) | Sup

---
## Step 3 — Rule-computation speedup table

In [10]:
try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

def total_rule_compute_ms(records):
    return sum(r['supportMs'] + r['confidenceMs'] for r in records)

compute_ms_r = total_rule_compute_ms(records_r)
compute_ms_c = total_rule_compute_ms(records_c)
compute_ms_g = total_rule_compute_ms(records_g) if records_g else None

rows = [
    {'Implementation': 'Rewritten C++',
     'Rule compute time (ms)': f'{compute_ms_r:.3f}',
     'Speedup vs Rewritten': '1.0x'},
    {'Implementation': 'Optimized CPU',
     'Rule compute time (ms)': f'{compute_ms_c:.3f}',
     'Speedup vs Rewritten': f'{compute_ms_r/compute_ms_c:.2f}x' if compute_ms_c else 'N/A'},
]
if compute_ms_g is not None:
    rows.append({
        'Implementation': 'GPU (CUDA)',
        'Rule compute time (ms)': f'{compute_ms_g:.3f}',
        'Speedup vs Rewritten': f'{compute_ms_r/compute_ms_g:.2f}x' if compute_ms_g else 'N/A'
    })
else:
    rows.append({
        'Implementation': 'GPU (CUDA)',
        'Rule compute time (ms)': 'N/A (no GPU)',
        'Speedup vs Rewritten': 'N/A'
    })

print('Speedup is based only on rule computation: sum(supportMs + confidenceMs).')
print('Wall time / initialization / graph loading / indexing / CUDA startup are excluded.\n')

if HAS_PANDAS:
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df.to_string(index=False))
else:
    header = f'{"Implementation":<22} {"Rule compute (ms)":>20} {"Speedup":>20}'
    print(header)
    print('-' * len(header))
    for r in rows:
        print(f'{r["Implementation"]:<22} {r["Rule compute time (ms)"]:>20} {r["Speedup vs Rewritten"]:>20}')

# -- Per-body-atom cross-impl comparison --------------------------------------
from collections import defaultdict

def group_by_atoms(records, field):
    g = defaultdict(list)
    for r in records:
        n = atom_counts.get(r['ruleId'], '?')
        if isinstance(n, int):
            g[n].append(r[field])
    return g

atom_ns = sorted(set(atom_counts.values()))

for field, label in [('confidenceMs', 'Avg Confidence ms/rule'), ('supportMs', 'Avg Support ms/rule')]:
    print(f'\n-- {label} by body-atom count --')
    header = f'  {"Atoms":<8}' + f'{"Rewritten":>14}' + f'{"Opt CPU":>12}'
    if records_g:
        header += f'{"GPU":>10}'
    print(header)
    print('  ' + '-' * (len(header) - 2))
    for n in atom_ns:
        g_r = group_by_atoms(records_r, field).get(n, [])
        g_c = group_by_atoms(records_c, field).get(n, [])
        avg_r = f'{sum(g_r)/len(g_r):.3f}' if g_r else 'N/A'
        avg_c = f'{sum(g_c)/len(g_c):.3f}' if g_c else 'N/A'
        row = f'  {n:<8}{avg_r:>14}{avg_c:>12}'
        if records_g:
            g_g = group_by_atoms(records_g, field).get(n, [])
            avg_g = f'{sum(g_g)/len(g_g):.3f}' if g_g else 'N/A'
            row += f'{avg_g:>10}'
        print(row)


Speedup is based only on rule computation: sum(supportMs + confidenceMs).
Wall time / initialization / graph loading / indexing / CUDA startup are excluded.



'Implementation Rule compute time (ms) Speedup vs Rewritten\n Rewritten C++             106785.000                 1.0x\n Optimized CPU              60497.504                1.77x\n    GPU (CUDA)                301.028              354.73x'


-- Avg Confidence ms/rule by body-atom count --
  Atoms        Rewritten     Opt CPU       GPU
  --------------------------------------------
  1               10.500       5.705     0.037
  2             8520.750    1510.921     0.950
  3            18046.250   13492.577    45.699

-- Avg Support ms/rule by body-atom count --
  Atoms        Rewritten     Opt CPU       GPU
  --------------------------------------------
  1               14.000       7.671     0.028
  2               38.250      34.482     0.283
  3               78.750      79.708    28.292


---
## Step 4 — JSON output demo (CPU → Scala integration)

Shows the `--json` flag that the Scala integration layer uses to communicate with the CPU binary.

In [11]:
print('Running CPU binary with --json flag ...')
result = subprocess.run(
    f'{DIR_CPU}/rdf_rules_test {TTL_FILE} {RULE_FILE_JSON} --json',
    shell=True, capture_output=True, text=True
)

print('stderr (timing info, kept separate from JSON):')
print(result.stderr.strip())
print()
print('stdout (clean JSON, ready for Scala to parse):')
print(result.stdout.strip())

# Parse and pretty-print
rules_json = json.loads(result.stdout)
print()
print('Parsed in Python successfully:')
for r in rules_json:
    print(f'  Rule {r["ruleId"]}: support={r["support"]}  confidence={r["confidence"]:.4f}')

Running CPU binary with --json flag ...
stderr (timing info, kept separate from JSON):
total_ms=60609.4

stdout (clean JSON, ready for Scala to parse):
[
  {
    "ruleId": 1,
    "rule": "( ?0 https://example.org/relation/interacts_with ?1 ) => ( ?1 https://example.org/relation/interacts_with ?0 )",
    "support": 820,
    "headSize": 100000,
    "headSupport": 100000,
    "headCoverage": 0.0082,
    "bodySize": 100000,
    "confidence": 0.0082,
    "supportMs": 6.23608,
    "confidenceMs": 6.27938
  },
  {
    "ruleId": 2,
    "rule": "( ?0 https://example.org/relation/interacts_with ?1 ) => ( ?0 https://example.org/relation/interacts_with ?1 )",
    "support": 0,
    "headSize": 100000,
    "headSupport": 100000,
    "headCoverage": 0,
    "bodySize": 100000,
    "confidence": 0,
    "supportMs": 11.5272,
    "confidenceMs": 6.34589
  },
  {
    "ruleId": 3,
    "rule": "( ?0 https://example.org/relation/interacts_with ?1 ) ^ ( ?1 https://example.org/relation/interacts_with ?2 ) => (

---
## Step 5 — Run the Scala integration demo (Colab only)

Installs `scala-cli` and runs `demo/RulesDemo.scala`.
This shows the exact pattern that would be used inside RDFRules.

> **On VSE JupyterHub**: skip this cell — scala-cli cannot be installed there.
> The GPU path requires the GPU Colab runtime.

In [12]:
if not IS_COLAB:
    print('Not on Colab — skipping scala-cli installation.')
else:
    if not has_command('scala-cli'):
        print('Installing scala-cli ...')
        run('curl -fL https://github.com/Virtuslab/scala-cli/releases/latest/download/scala-cli-x86_64-pc-linux.gz | gunzip > /usr/local/bin/scala-cli && chmod +x /usr/local/bin/scala-cli')
    print('scala-cli version:', subprocess.run(['scala-cli', '--version'], capture_output=True, text=True).stdout.strip())

Not on Colab — skipping scala-cli installation.


In [13]:
if not IS_COLAB:
    print('Not on Colab — skipping Scala demo.')
elif not has_command('scala-cli'):
    print('scala-cli not available.')
else:
    print('Running RulesDemo.scala ...')
    demo_script = BASE / 'demo' / 'RulesDemo.scala'
    run(f'scala-cli run {demo_script} -- {TTL_FILE} {RULE_FILE_JSON}')

Not on Colab — skipping Scala demo.


---
## Step 6 — GPU on full dataset (gpu-test.ttl, 1 000 000 triples)

Runs the same benchmark rule on the full 1M-triple graph to show how the GPU scales.

In [14]:
TTL_FULL = DATA_DIR / 'gpu-test.ttl'

if not HAS_GPU:
    print('No GPU available — skipping.')
elif not TTL_FULL.exists():
    print(f'Full TTL not found: {TTL_FULL}')
else:
    print(f'Running GPU_03 on full 1M-triple dataset: {TTL_FULL.name}')
    cmd_gpu_full = f'{DIR_GPU}/bench_gpu {TTL_FULL} {RULE_FILE_JSON}'
    min_ms_gf, avg_ms_gf, out_gf = benchmark(cmd_gpu_full, runs=3)
    total_gf = extract_total_ms(out_gf)

    summary_start = out_gf.find('=== Timing summary ===')
    print(out_gf[summary_start:] if summary_start != -1 else out_gf[-3000:])
    print(f'Wall time: min={min_ms_gf:.0f} ms  avg={avg_ms_gf:.0f} ms  (reported total={total_gf} ms)')

    if min_ms_g is not None:
        scale = min_ms_gf / min_ms_g
        ratio = 1_000_000 / 100_000
        print(f'\n100k triples: {min_ms_g:.0f} ms → 1M triples: {min_ms_gf:.0f} ms')
        print(f'Scale factor: {scale:.2f}x  (dataset is {ratio:.0f}x larger)')


Running GPU_03 on full 1M-triple dataset: gpu-test.ttl
=== Timing summary ===
Indexing time:          834.551 ms
CSR build+upload time:  1800.05 ms
Rule parsing time:      127.007 ms
Support counting time:  5875.01 ms
Confidence counting time: 11054.2 ms
Total time:             19690.9 ms
Warp-coop rules:        8  (threshold maxfan > 64)

=== Per-rule SUPPORT statistics (10 rules) ===
Average: 583.818 ms
Median:  4.26741 ms
Std Dev: 1714.94 ms
Min:     0.027936 ms
Max:     5728.43 ms
P95:     5728.43 ms

=== Per-rule CONFIDENCE statistics (10 rules) ===
Average: 1101.76 ms
Median:  12.6989 ms
Std Dev: 1472.73 ms
Min:     0.023264 ms
Max:     3781.83 ms
P95:     3781.83 ms

=== SUPPORT statistics by body size ===
Body size 1 (2 rules): avg=0.04632 ms, median=0.04632 ms, min=0.027936 ms, max=0.064704 ms, P95=0.064704 ms
Body size 2 (4 rules): avg=2.59716 ms, median=0.983248 ms, min=0.917472 ms, max=7.50467 ms, P95=7.50467 ms
Body size 3 (4 rules): avg=1456.92 ms, median=40.0038 ms, min=

---
## Memory Usage — Peak RAM and VRAM per implementation

In [15]:
print('Measuring peak memory usage (this re-runs each implementation silently) ...\n')

ram_r, vram_r, _ = measure_peak_memory(f'{DIR_REWRITTEN}/rdf_rules_rewritten {TTL_FILE} {RULE_FILE_JSON}')
print(f'Rewritten C++  — RAM: {ram_r:>8,.0f} MB   VRAM: {vram_r:>6,.0f} MB')

ram_c, vram_c, _ = measure_peak_memory(f'{DIR_CPU}/rdf_rules_test {TTL_FILE} {RULE_FILE_JSON}')
print(f'Optimized CPU  — RAM: {ram_c:>8,.0f} MB   VRAM: {vram_c:>6,.0f} MB')

if HAS_GPU:
    ram_g, vram_g, _ = measure_peak_memory(f'{DIR_GPU}/bench_gpu {TTL_FILE} {RULE_FILE_JSON}')
    print(f'GPU (CUDA)     — RAM: {ram_g:>8,.0f} MB   VRAM: {vram_g:>6,.0f} MB')
else:
    print('GPU (CUDA)     — skipped (no GPU)')

Measuring peak memory usage (this re-runs each implementation silently) ...

Rewritten C++  — RAM:    3,082 MB   VRAM:      0 MB
Optimized CPU  — RAM:       16 MB   VRAM:      0 MB
GPU (CUDA)     — RAM:      258 MB   VRAM:    533 MB
